# Gemma 4 | Bold Shell First

A new experiment: short edit-first instructions, four tools, shell-based editing,
and temperature 1.0 / top-k 64. Keep the 31B model, thinking with replay, and the
0.15 base's 4.5-minute / 28-call / 80-turn limits. No measured score is claimed.

Credits: Matterhorn3838's [SuperAgent](https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent)
and our 0.15 budget baseline; MUHAMMED IDRIS BIN YACOOB's v24 sampling settings
([published replica](https://www.kaggle.com/code/cosmorin/gemma-4-idris-v24-e07-faithful-replica));
[Jayaprakash Sundararaj](https://www.jayaprakash.net/blog/2026-09-24-gemma-4-developer-agent-on-kaggle/)
for the failed-edit diagnosis. The prompt is newly written. See README.md for evidence,
limitations, exact base diff, and the second-ranked idea.

Run All on CPU, internet off. This notebook only builds and checks a four-file
`submission.zip`; it does not run inference or submit to Kaggle. The packaging cell
uses only the standard library and runs before optional PyYAML configuration checks.
Phase deadlines are model instructions. The 12-hour global limit also includes setup;
120 tasks at the agent cap consume 9 hours, leaving 3 hours for overhead.


In [1]:
from pathlib import Path
import hashlib
import os
import zipfile

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [2]:
# Keep the bundle literal and inspectable.
FILES = {
    'agent.yaml': """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - get_status
  - submit_patch
generate_content_config: !include configs/sampling.yaml
""",
    'configs/sampling.yaml': """temperature: 1.0
top_p: 0.95
top_k: 64
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    'eval_config.yaml': """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 4.5
  max_turns: 80
""",
    'prompts/system.md': """Fix the issue in /workspace with a small, correct source patch. Match the requested names, defaults, types, messages and edge cases exactly. Preserve unrelated behavior and update parallel implementations when needed.

You have 4.5 minutes and 28 charged tool calls. Use get_status initially and after editing. Aim for the first justified source edit within 8 charged calls or 90 seconds, whichever comes first; this is an instruction, not an enforced phase. Locate with a specific git grep, read the implementation and nearest caller or test, then edit. Do not spend the opening on reproductions or broad exploration. If uncertain, use the strongest source evidence to make a focused candidate fix, then refine it.

Use run_command for edits: execute Python through a single-quoted heredoc. For an existing file, read its exact text, assert the old snippet occurs exactly once, replace that snippet, and write the result. Preserve encoding and line endings. Do not rewrite a whole existing file from memory. If an assertion fails, reread the actual region and change the snippet; never repeat the identical failing command. Create a source file only when the issue needs it.

Keep search and read output short. Do not edit tests, test configuration, or unrelated files. Do not install packages or search outside /workspace. Use inline Python for scratch checks.

After the edit, run one issue-specific assertion or focused existing test with a shell timeout of at most 30 seconds, only if get_status shows more than 75 seconds remaining. Never run the full suite. For an issue-relevant failure, make one correction and rerun only if time permits. Missing fixtures or unrelated import errors are inconclusive; do not repair the environment.

At 45 seconds remaining, stop exploration and testing. Review git status --short, git diff --check and git diff; leave only the intended source changes. Call submit_patch once as your final action, even if a local check could not run. An empty patch cannot solve the issue; a non-empty patch still needs to be correct.
""",
}
SYSTEM_PROMPT = FILES['prompts/system.md']


In [3]:
# Fixed member order, timestamps and permissions make packaging reproducible.
with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert len(archive.infolist()) == 4
    assert set(archive.namelist()) == set(FILES)
    for relative, content in FILES.items():
        assert archive.read(relative) == content.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Four members verified; inference and leaderboard scoring were not run.')


Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/submission_bold_shell_first/submission.zip
SHA-256: a7b8727b94dfbe6fc3f9fb17c00a8f371466a60d0f22a2ef7796c0c95fca9175
Four members verified; inference and leaderboard scoring were not run.


In [4]:
# Optional local configuration checks; ZIP creation has already completed.
import yaml

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=IncludeLoader)
sampling = agent['generate_content_config']
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['tools'] == ['run_command', 'read_file', 'get_status', 'submit_patch']
assert sampling == {'temperature': 1.0, 'top_p': 0.95, 'top_k': 64,
                    'max_output_tokens': 8192,
                    'thinking_config': {'thinking_budget': 4096, 'include_thoughts': True}}
assert evaluation == {'timeout_seconds': 240, 'max_tool_calls': 28,
                      'max_time_minutes': 4.5, 'max_turns': 80}
assert set(FILES) == {'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md'}
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT
assert len(SYSTEM_PROMPT.split()) < 400
assert '8 charged calls or 90 seconds' in SYSTEM_PROMPT
assert 120 * evaluation['max_time_minutes'] == 540
print('Config checks passed: four tools; thinking retained; 9-hour agent ceiling for 120 tasks.')
print('Global completion and repair quality still require the official scorer.')


Config checks passed: four tools; thinking retained; 9-hour agent ceiling for 120 tasks.
Global completion and repair quality still require the official scorer.
